##CELL 1: SETUP & CONFIGURATION

In [0]:
import logging
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, when, rand, current_date, date_sub, lit, expr
from pyspark.sql.types import IntegerType

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Utility to safely pull variables from Databricks Job parameters
def get_param(param_name: str, default_value: str) -> str:
    try:
        return dbutils.widgets.get(param_name)
    except Exception:
        return default_value

# Pull from orchestrator (variables.yml) or fallback to local defaults
CATALOG = get_param("catalog_name", "main")
SCHEMA = get_param("schema_name", "fixkart_prd")
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.landing_bookings"

NUM_RECORDS = 50000
WRITE_MODE = "overwrite"

logger.info(f"Initialized Mock Generator for Target: {TARGET_TABLE}")

## CELL 2: MOCK DATA GENERATION LOGIC


In [0]:
def create_mock_landing_data(spark: SparkSession, num_rows: int) -> DataFrame:
    logger.info(f"Generating {num_rows} raw mock records...")
    
    try:
        df_raw = (
            spark.range(0, num_rows)
            .withColumnRenamed("id", "work_order_id")
            .withColumn("customer_id", expr("uuid()")) 
            .withColumn("technician_id", (rand() * 500).cast(IntegerType()) + 1000)
            .withColumn(
                "city", 
                when(rand() < 0.40, lit("Mumbai"))
                .when(rand() < 0.70, lit("Bengaluru"))
                .when(rand() < 0.85, lit("Pune"))
                .when(rand() < 0.95, lit("Delhi"))
                .otherwise(lit("Hyderabad"))
            )
            .withColumn("booking_date", date_sub(current_date(), (rand() * 90).cast(IntegerType())))
            .withColumn("status_rand", rand())
            .withColumn(
                "status", 
                when(col("status_rand") <= 0.14, lit("FAILED"))
                .when(col("status_rand") <= 0.86, lit("COMPLETED"))
                .otherwise(lit("CANCELLED"))
            )
            .withColumn("failure_reason_rand", rand())
            .withColumn(
                "failure_reason", 
                when(col("status") != "FAILED", lit(None))
                .when(col("failure_reason_rand") <= 0.25, lit("Customer_Forgot_Slot"))
                .when(col("failure_reason_rand") <= 0.40, lit("Customer_Not_Home"))
                .when(col("failure_reason_rand") <= 0.45, lit("Customer_Unreachable"))
                .when(col("failure_reason_rand") <= 0.65, lit("Tech_Previous_Job_Overrun"))
                .when(col("failure_reason_rand") <= 0.80, lit("Tech_Transit_Delay"))
                .when(col("failure_reason_rand") <= 0.90, lit("Reschedule_Not_Synced"))
                .otherwise(lit("Unconfirmed_Assignment"))
            )
            .withColumn(
                "reschedule_count", 
                when(rand() < 0.80, lit(0))
                .when(rand() < 0.95, lit(1))
                .otherwise(lit(2))
            )
            .drop("status_rand", "failure_reason_rand")
        )
        
        logger.info("Mock data generation completed successfully.")
        return df_raw
        
    except Exception as e:
        logger.error(f"Error during mock data generation: {str(e)}")
        raise

##CELL 3: DELTA LAKE WRITE UTILITIES

In [0]:
def write_to_delta(df: DataFrame, table_name: str, write_mode: str) -> None:
    logger.info(f"Writing data to Delta table '{table_name}' using mode '{write_mode}'...")
    try:
        df.write.format("delta").mode(write_mode).saveAsTable(table_name)
        logger.info(f"Successfully wrote records to {table_name}.")
    except Exception as e:
        logger.error(f"Failed to write to Delta table {table_name}: {str(e)}")
        raise

##CELL 4: ORCHESTRATION & EXECUTION

In [0]:
if 'spark' in locals() or 'spark' in globals():
    try:
        logger.info("--- Starting Pipeline Orchestration ---")
        
        landing_df = create_mock_landing_data(spark=spark, num_rows=NUM_RECORDS)
        write_to_delta(df=landing_df, table_name=TARGET_TABLE, write_mode=WRITE_MODE)
        
        logger.info("--- Pipeline Completed Successfully ---")
        
    except Exception as main_e:
        logger.error("Pipeline execution aborted due to an error.")
else:
    logger.warning("SparkSession not found.")

##CELL 5: VALIDATION

In [0]:
# Read back from the Delta table to ensure Unity Catalog registration was successful
display(spark.table(TARGET_TABLE).limit(20))